# Physics-Informed Neural Network (PINN)

Train a network to solve a differential equation by penalizing how much it violates the
equation, instead of fitting it to precomputed solution data. We use the damped harmonic
oscillator - the standard first PINN example - because it has a known closed-form solution,
so we can measure exactly how close the network gets.

$$m u'' + c u' + k u = 0, \quad u(0) = 1, \quad u'(0) = 0$$

with underdamped parameters ($c^2 < 4mk$) so the true solution oscillates while decaying:

$$u(t) = e^{-\delta t}\left(\cos(\omega t) + \frac{\delta}{\omega}\sin(\omega t)\right), \quad
\delta = \frac{c}{2m}, \quad \omega = \sqrt{\frac{k}{m} - \delta^2}$$

Note: the oscillation frequency matters a lot here. A vanilla PINN like this one struggles on
high-frequency solutions (a known limitation - MLPs are biased toward learning low-frequency
functions first). The parameters below are chosen to be gentle for that reason; try increasing
`k` and see training get much harder.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

m, c, k = 1.0, 0.4, 9.0
delta = c / (2 * m)
omega0 = (k / m) ** 0.5
omega = (omega0**2 - delta**2) ** 0.5
t_max = 4.0
assert delta**2 < omega0**2, "parameters must be underdamped"


def analytic(t):
    return torch.exp(-delta * t) * (torch.cos(omega * t) + (delta / omega) * torch.sin(omega * t))

## The network and the physics-informed loss

The network maps time `t` to a predicted `u(t)`. The loss has two parts:

- **Physics residual**: at randomly sampled "collocation" points, compute $u''$ and $u'$ via
  autograd and penalize $m u'' + c u' + k u$ being nonzero - this is what makes it
  physics-*informed* rather than data-fitted.
- **Initial condition**: penalize $u(0) \neq 1$ and $u'(0) \neq 0$ directly.

No solution data is used anywhere - the network only ever sees the equation itself.

In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden=64, layers=4):
        super().__init__()
        seq = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(layers - 1):
            seq += [nn.Linear(hidden, hidden), nn.Tanh()]
        seq += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*seq)

    def forward(self, t):
        return self.net(t)


model = PINN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
n_collocation = 128

for epoch in range(4001):
    opt.zero_grad()

    t_c = (torch.rand(n_collocation, 1, device=device) * t_max).requires_grad_(True)
    u_c = model(t_c)
    du_c = torch.autograd.grad(u_c, t_c, grad_outputs=torch.ones_like(u_c), create_graph=True)[0]
    d2u_c = torch.autograd.grad(du_c, t_c, grad_outputs=torch.ones_like(du_c), create_graph=True)[0]
    physics_loss = torch.mean((m * d2u_c + c * du_c + k * u_c) ** 2)

    t0 = torch.zeros(1, 1, device=device, requires_grad=True)
    u0 = model(t0)
    du0 = torch.autograd.grad(u0, t0, grad_outputs=torch.ones_like(u0), create_graph=True)[0]
    ic_loss = (u0 - 1.0) ** 2 + (du0 - 0.0) ** 2

    loss = physics_loss + 10.0 * ic_loss.squeeze()
    loss.backward()
    opt.step()

    if epoch % 800 == 0:
        print(f"epoch {epoch:5d}  loss {loss.item():.4f}")

## Compare against the analytic solution

This is the real test: the network was never shown `analytic(t)` during training, only the
differential equation. If the physics loss did its job, the two should line up closely.

In [ ]:
t_test = torch.linspace(0, t_max, 200, device=device).reshape(-1, 1)
with torch.no_grad():
    u_pred = model(t_test).squeeze()
u_true = analytic(t_test.squeeze())

mse = torch.mean((u_pred - u_true) ** 2).item()
max_err = torch.max(torch.abs(u_pred - u_true)).item()
print(f"MSE vs. analytic solution: {mse:.6f}")
print(f"max absolute error:        {max_err:.4f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(t_test.cpu().squeeze(), u_true.cpu(), label="analytic", linewidth=2)
plt.plot(t_test.cpu().squeeze(), u_pred.cpu(), "--", label="PINN", linewidth=2)
plt.xlabel("t")
plt.ylabel("u(t)")
plt.legend()
plt.title("Damped harmonic oscillator: PINN vs. analytic solution")
plt.show()

## Next steps

- Increase `k` (raise the oscillation frequency) and watch the fit degrade - this is the
  "spectral bias" limitation mentioned above, a real and active research topic.
- Swap in a different ODE or a simple PDE (e.g. the 1D heat equation - compare against the
  `neural-operator.ipynb` notebook, which solves a related problem a completely different way).
- Try a sinusoidal/Fourier feature input embedding instead of feeding `t` directly - a common
  fix for the frequency issue above.